### Test de coherencia de genero

In [8]:
import faiss
import pandas as pd

INDEX_PATH = "../data/processed/faiss.index"
INDEX_META_PATH = "../data/processed/index_metadata.csv"

index=faiss.read_index(INDEX_PATH)
metadata=pd.read_csv(INDEX_META_PATH)
metadata.head()
k=6

In [20]:
from sklearn.metrics import jaccard_score
from sklearn.preprocessing import MultiLabelBinarizer

N_SAMPLES=500
# tomamos los mismos 500 samples siempre
sample=metadata.sample(N_SAMPLES, random_state=42).reset_index(drop=True)
metadata['genre_list'] = metadata['genres'].fillna('').apply(lambda x: [g.strip().lower() for g in x.split(", ")] if x else [])


mlb = MultiLabelBinarizer()
generos_binarios = mlb.fit_transform(metadata['genre_list'])


import numpy as np

scores_totales = []

for idx, row in sample.iterrows():
    # 1. Recuperar vector y buscar en FAISS
    vector = index.reconstruct(idx)
    # k=6 para saltar la misma película
    socres, indices = index.search(vector.reshape(1, -1), k)
    # 2. Obtener los vectores binarios de las recomendaciones (excluyendo el primero)
    indices_recomendados = indices[0][1:]
    
    # El vector binario de la película original (query)
    y_true = generos_binarios[idx] 
    
    # Los vectores binarios de las 5 recomendadas
    y_preds = generos_binarios[indices_recomendados]
    
    # 3. Calcular Jaccard para este set de 5 películas
    # 'samples' calcula el Jaccard para cada par y devuelve el promedio
    # Necesitamos repetir y_true para que tenga la misma forma que y_preds
    y_true_repeated = np.tile(y_true, (len(y_preds), 1))
    
    score_pelicula = jaccard_score(y_true_repeated, y_preds, average='samples')
    scores_totales.append(score_pelicula)

print(f"Promedio Jaccard (scikit-learn): {np.mean(scores_totales):.4f}")

Promedio Jaccard (scikit-learn): 0.3587
